# Dunnhumby M1 전파 깊이별 고정 CLV 진단

기존 seed 42 M1 체크포인트를 재학습하지 않고, 같은 layer-0 파라미터에서 `E⁽⁰⁾`, `(E⁽⁰⁾+E⁽¹⁾)/2`, `(E⁽⁰⁾+E⁽¹⁾+E⁽²⁾)/3`을 비교합니다.

- 평가구간: 역사적 개발구간 684~690일
- 평가과업: 학습기간 구매상품을 제외한 신규상품 추천
- 분석단위: 전체, 저·중·고 고정 CLV, N/V 4유형, 고CLV 내부 V우세·균형·N우세
- 주의: 학습·체크포인트 선택·최종 test·holdout 평가는 없습니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

REVIEWED_SHA = 'c7e815d'
%cd /content
!rm -rf /content/clv-m2-lightgcn-runner
!git clone -q https://github.com/jung-un/clv-m2-lightgcn-runner.git
%cd /content/clv-m2-lightgcn-runner
!git checkout -q $REVIEWED_SHA
import subprocess
assert subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], text=True).strip().startswith(REVIEWED_SHA)

In [ ]:
import importlib
import inspect
import json
import torch
import lightgcn_clv_layer_depth_diagnostic as depth_diagnostic
depth_diagnostic = importlib.reload(depth_diagnostic)

assert torch.cuda.is_available(), '런타임 유형에서 GPU를 선택하세요.'
parity_default = inspect.signature(
    depth_diagnostic.assert_full_view_parity
).parameters['atol'].default
assert parity_default == 1e-5, (
    f'수정 전 진단 모듈이 로드되었습니다: atol={parity_default}'
)
cfg = depth_diagnostic.configure_layer_depth_diagnostic('dunnhumby')
print(json.dumps(depth_diagnostic.preflight_summary(cfg), ensure_ascii=False, indent=2))

In [ ]:
paths = depth_diagnostic.run_layer_depth_diagnostic(cfg)

In [ ]:
import pandas as pd
from IPython.display import display

metrics = pd.read_csv(paths['view_metrics_csv'])
comparison = pd.read_csv(paths['comparison_csv'])
core = [
    'recall@10', 'ndcg@10', 'recall@20', 'ndcg@20',
    'recall@50', 'ndcg@50',
    'price_purchase_amount_weighted_hit@10',
]
groups = metrics.group_type.isin(['overall', 'fixed_clv_segment', 'high_clv_composition'])
print('1) 전파 깊이별 전체·CLV 구간 성과')
display(metrics.loc[groups, ['view', 'group_type', 'group', 'n_users', *core]])

focus = comparison.metric.isin(core) & comparison.group_type.isin(
    ['overall', 'fixed_clv_segment', 'high_clv_composition']
)
print('2) 표준 M1(layer 0·1·2 평균) 대비 변화')
display(comparison.loc[focus].sort_values(['view', 'group_type', 'group', 'metric']))
print('결과 파일:', paths)